# KataGo Human SL 分析

重現 `aicup/analyze_humansl.py` 的分析，輸出每局在各 profile 下的落子機率，以及每手損失（`val648` 模式不算損失）。

- runtime：T4 GPU
- 外部資源：KataGo `v1.18.2`、`b18c384nbt-humanv0`、`kata1-b28c512nbt-s13255194368-d5935380940`，授權見報告的外部資源一節
- Drive 的 `MyDrive/aicup-2026/` 要有 `training-v1.1/` 的 10 個 CSV（官方訓練集 v1.1，含 `HD[n]`）與 `katago/` 的 2 個模型
- `val648` 模式另外要有 `humansl-analysis/val-players.csv`，欄位是 `player_id`、`rank`
- `val798` 模式另外要有 `humansl-analysis/val-players-798.csv`，由 `python -m aicup.val_players <val_probs.npz> --min-games 5 --out humansl-analysis/val-players-798.csv` 產生
- `val648_opp` 模式另外要有 `humansl-analysis/val-players.csv`，以及快速設定檢查用的 `humansl-analysis/val648-p3.jsonl`
- `test` 模式另外要有 `tests/rank_prediction_test_{public,private}.csv`，以及快速設定檢查用的 `humansl-analysis/val648-p3.jsonl`、`val-players.csv`
- 規則、貼目、讓子依官方 Q&A，預設 `japanese`、6.5 目，`HD` 的處理見 `parse_game`
- Colab Secrets 要有唯讀的 `GH_TOKEN`

Drive 的做法：

- 不跑 `aicup.colab_setup`。原因：它會讀 `rank-features/`，曾讓 Drive 掛載斷線，這份分析用不到特徵。
- CSV 與模型先複製到 `/content`，分析期間不讀 Drive。
- 結果寫在 `/content`，每批結束後複製到 Drive。原因：直接寫 Drive 時，檔案開著的期間資料可能沒有同步，斷線後會遺失。
- 斷線時最多損失最近一批。重新開 runtime，依序執行所有 cell 就會接續。
- 分析中不要中斷 cell。

## 參數

`MODE` 選 `pilot`、`visits`、`main`、`val648`、`val798`、`val648_opp`、`test`。

- `val648`：CNN val 切分中局數 20 以上的 648 位玩家，每人 20 局，3 個 profile，不算損失
- `val798`：CNN val 切分中局數 5 以上的 798 位玩家，每人最多 20 局，局數 20 以上的玩家抽到的局和 `val648` 相同，3 個 profile，不算損失
- `val648_opp`：和 `val648` 相同的 12,960 局，改評對手的手（`--side opponent`）
- `test`：競賽二的 800 題測試題，共 10,000 局，3 個 profile，不算損失
- `FAST`：只用於 `val648_opp`、`test`。主模型改用 human 模型並加大 GPU 批次；不算損失時主模型的輸出用不到。分析前先用 val648 的 40 局比對結果，不一致時自動改回原設定

In [ ]:
MODE = 'test'
BATCH = 500
FAST = True

DRIVE = '/content/drive/MyDrive/aicup-2026'
RUNS = {  # (參數, 輸出檔名, 總局數)
    'pilot': ('--players-per-rank 1 --games-per-player 10', 'pilot-100-hd-v1.jsonl', 100),
    'visits': ('--players-per-rank 1 --games-per-player 10 --eval-visits 16', 'pilot-100-hd-v16.jsonl', 100),
    'main': ('--players-per-rank 30 --games-per-player 20', 'pilot-6000-hd.jsonl', 6000),
    'val648': ('--players-file /content/val-players.csv --games-per-player 20 '
               '--profiles rank_11k rank_1d rank_6d --no-eval', 'val648-p3.jsonl', 12960),
    'val798': ('--players-file /content/val-players-798.csv --games-per-player 20 --min-games 5 '
               '--profiles rank_11k rank_1d rank_6d --no-eval', 'val798-p3.jsonl', 14870),
    'val648_opp': ('--players-file /content/val-players.csv --games-per-player 20 '
                   '--profiles rank_11k rank_1d rank_6d --no-eval --side opponent', 'val648-p3-opp.jsonl', 12960),
    'test': ('--test-dir /content/tests --profiles rank_11k rank_1d rank_6d --no-eval', 'test-p3.jsonl', 10000),
}
FAST_ARGS = ('--model b18c384nbt-humanv0.bin.gz --group 32 --override-config '
             'numAnalysisThreads=128,numSearchThreadsPerAnalysisThread=1,nnMaxBatchSize=128')
ARGS, OUT_NAME, TOTAL = RUNS[MODE]

## 取得程式

In [ ]:
import os, subprocess
from google.colab import drive, userdata

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'


def git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr.replace(userdata.get('GH_TOKEN'), '***'))
        raise SystemExit('git %s 失敗' % args[0])


if not os.path.exists(REPO):
    git('clone', '-q', 'https://%s@github.com/Jx-study/aicup-2026-go.git' % userdata.get('GH_TOKEN'), REPO)
os.chdir(REPO)
git('pull', '-q')
!pip install -q sgfmill
!git log --oneline -1
!nvidia-smi -L
os.environ['DRIVE'], os.environ['OUT_NAME'], os.environ['MODE'] = DRIVE, OUT_NAME, MODE

## 複製資料並安裝 KataGo

要看到 `10`、兩個模型與 `katago`，`val648` 模式還要看到 `val-players.csv`，`val798` 模式還要看到 `val-players-798.csv`，`val648_opp` 模式還要看到 `val648-p3.jsonl`，`test` 模式還要看到 2 個測試 CSV 與 `val648-p3.jsonl`。Drive 上已有結果時會複製回本地，接續分析。

In [ ]:
%%bash
mkdir -p /content/training /content/models /content/katago
cp "$DRIVE"/training-v1.1/train_*.csv /content/training/
cp "$DRIVE"/katago/*.bin.gz /content/models/
for f in "$DRIVE"/humansl-analysis/val-players*.csv; do
    if [ -f "$f" ]; then cp "$f" /content/; fi
done
if [ "$MODE" = test ]; then
    mkdir -p /content/tests
    cp "$DRIVE"/tests/rank_prediction_test_*.csv /content/tests/
    cp "$DRIVE/humansl-analysis/val648-p3.jsonl" /content/
    ls -la /content/tests /content/val648-p3.jsonl
fi
if [ "$MODE" = val648_opp ]; then
    cp "$DRIVE/humansl-analysis/val648-p3.jsonl" /content/
    ls -la /content/val648-p3.jsonl
fi
if [ -f "$DRIVE/humansl-analysis/$OUT_NAME" ]; then
    cp "$DRIVE/humansl-analysis/$OUT_NAME" /content/
fi
cd /content/katago
ZIP=katago-v1.18.2-cuda12.8-cudnn9.8.0-linux-x64.zip
wget -q https://github.com/lightvector/KataGo/releases/download/v1.18.2/$ZIP
unzip -qjo $ZIP
chmod +x katago
ls /content/training | wc -l
ls -la /content/models /content/katago/katago analysis_example.cfg /content/*.csv

## 快速設定檢查

只在 `val648_opp` 或 `test` 模式且 `FAST = True` 時執行。用快速設定重跑 val648 的前 40 局，和 `val648-p3.jsonl` 比對每個 profile 的每手平均 log 機率，也就是 stacking 用的特徵。比對一律評目標玩家的手，快速設定只換主模型與批次。40 局都要在參考檔中，且最大差小於 0.01 才採用。最後一行印出分析要用的參數。

In [ ]:
import json, time
import numpy as np

if MODE in ('val648_opp', 'test') and FAST:
    N = 40
    CHECK = '/content/fastcheck.jsonl'
    !rm -f {CHECK}
    t = time.time()
    !python -m aicup.analyze_humansl --csv '/content/training/train_*.csv' --model-dir /content/models --players-file /content/val-players.csv --games-per-player 20 --profiles rank_11k rank_1d rank_6d --no-eval --limit {N} {FAST_ARGS} --out {CHECK} 2>&1 | tail -3
    sec = (time.time() - t) / N
    def mean_logp(probs):
        p = np.array(probs, dtype=float)
        return np.nanmean(np.log(np.clip(p, 1e-6, None)), axis=1)  # 同 fit_humansl 的特徵

    ref = {}
    for line in open('/content/val648-p3.jsonl'):
        r = json.loads(line)
        ref[r['key']] = mean_logp(r['probs'])
    rows = [json.loads(l) for l in open(CHECK)] if os.path.exists(CHECK) else []
    rows = [r for r in rows if r['key'] in ref]
    diff = max((np.max(np.abs(mean_logp(r['probs']) - ref[r['key']])) for r in rows), default=np.inf)
    print('快速設定：%d/%d 局可比對，含啟動每局 %.2f 秒，平均 log 機率的最大差 %.2g' % (len(rows), N, sec, diff))
    if len(rows) == N and diff < 0.01:
        ARGS = ARGS + ' ' + FAST_ARGS
        print('採用快速設定')
    else:
        print('不採用快速設定，改用原設定')
print(ARGS)

## 分析

每批結束會印出 Drive 上的行數，數字增加代表備份成功。輸出檔已有的行數以內的批次會跳過。

In [ ]:
LOCAL = '/content/' + OUT_NAME
LOG = '/content/%s.log' % MODE
BACKUP = DRIVE + '/humansl-analysis'
done = sum(1 for _ in open(LOCAL)) if os.path.exists(LOCAL) else 0
for n in sorted({*range(BATCH, TOTAL, BATCH), TOTAL}):
    if n <= done and n < TOTAL:  # 抽樣結果的前 n 局已在輸出檔內
        continue
    !python -m aicup.analyze_humansl --csv '/content/training/train_*.csv' --model-dir /content/models {ARGS} --limit {n} --out {LOCAL} 2>&1 | tee -a {LOG}
    !mkdir -p {BACKUP} && cp {LOCAL} {LOG} {BACKUP}/ && wc -l {BACKUP}/{OUT_NAME}

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣額度。

In [ ]:
from google.colab import drive, runtime

drive.flush_and_unmount()
runtime.unassign()